# model_inference

Final inference: the overall best architecture (**XGBoost** — Kaggle public 2448.73 / private 2515.09) is registered in the MLflow Model Registry as **`walmart-best-model`**, loaded back **from the registry**, and run directly on the raw, un-preprocessed `test.csv` — all preprocessing lives inside the registered sklearn pipeline. Post-processing (Christmas shift + no-Christmas holiday blend, w=0.75) matches the winning submission exactly.

In [9]:
# Kaggle bootstrap — attach competition data + the three MLFLOW secrets first.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow xgboost scikit-learn")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [10]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.mlflow_utils import setup_mlflow, REGISTRY_MODEL_NAME
from src.postprocess import apply_christmas_shift, blend_holiday_naive
from src.preprocessing import BLEND_HOLIDAY_WEEKS

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
setup_mlflow("Inference")

MLflow -> https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow | experiment: Inference


## Step 1 — Register the winner as `walmart-best-model` (one-time)

XGBoost won the architecture comparison on Kaggle, so its full pipeline (preprocessing + model) is promoted to the registry name the inference loads from. Skip this cell if the registered model already exists.

In [11]:
best_pipeline = mlflow.sklearn.load_model("models:/walmart-xgboost/latest")
with mlflow.start_run(run_name="Register_Best_Model"):
    mlflow.log_params({
        "source": "walmart-xgboost (best single model: Kaggle public 2448.73 / private 2515.09)",
        "post_processing": "christmas_shift + noXmas_blend w=0.75",
    })
    mlflow.sklearn.log_model(best_pipeline, name="model",
                             serialization_format="cloudpickle",
                             registered_model_name=REGISTRY_MODEL_NAME)
print("registered:", REGISTRY_MODEL_NAME)

2026/07/11 15:04:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'walmart-best-model'.
2026/07/11 15:05:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-best-model, version 1
Created version '1' of model 'walmart-best-model'.


🏃 View run Register_Best_Model at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/8/runs/7594ca89952e4c45b98d9727b79db00e
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/8
registered: walmart-best-model


## Step 2 — Load from the registry and predict the RAW test set

In [12]:
model = mlflow.sklearn.load_model(f"models:/{REGISTRY_MODEL_NAME}/latest")

raw_test = test[["Store", "Dept", "Date"]]          # RAW — no preprocessing here
sub = test[["Store", "Dept", "Date"]].copy()
sub["pred"] = model.predict(raw_test).astype(float)

sub = apply_christmas_shift(sub, pred_col="pred")
sub = blend_holiday_naive(sub, train, weight=0.75, holiday_dates=BLEND_HOLIDAY_WEEKS)

make_submission(sub, "pred", "submission.csv")
sub.head()

,Store,Dept,Date,pred
0,1,1,2012-11-02,41520.406250
1,1,1,2012-11-09,19775.841797
2,1,1,2012-11-16,19380.390625
3,1,1,2012-11-23,21189.487793
4,1,1,2012-11-30,27299.558594


Upload `submission.csv` to Kaggle:

```
kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission.csv -m "registry model (xgboost)"
```

Expected score: identical to the XGBoost submission (public ~2448.73) — same pipeline, same post-processing, loaded from the registry.